In [1]:
import os
import sys
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import pandas as pd
from tqdm import tqdm
import numpy as np
from scipy.stats import pearsonr, spearmanr
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from importlib import reload
import matplotlib.pyplot as plt
from suncalc import get_position, get_times
from datetime import datetime, timezone
from statsmodels.stats.multitest import multipletests

In [2]:
sys.path.append("../src")
import main

In [ ]:
reload(main)

In [4]:
temp = pd.read_csv('../data/raw_data/ta_vp_rh_60_min_2022_09_01_2024_08_31_gap_filled_new.csv')
temp['datetime']=pd.to_datetime(temp['datetime'])
temp['datetime_UTC'] = temp['datetime']
temp['value'] = temp['ta']
temp = temp[temp['type'] == 'measured']
temp['datetime_UTC'] = temp['datetime_UTC'].astype(str)
temp = temp.pivot(index='station_id', columns='datetime_UTC', values='value')
temp = temp.drop(['FREICH','FRWITT'], axis=0)
temp.index = temp.index.str[2:]

In [5]:
vars = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum', 'BuIBD', 'BuAdj', 'BuSWR_3D_median', 'BuHt_wmean','StrHW_median','SVF_3D_mean','SVF_3D_std', 'BuERI_mode', 'StrClo400_median','BuAre_median','BuVol_3D_median','BuEWA_3D_median','BuSWR_median','BuHt_max','StrHW_mean','BuERI_wmean']

In [ ]:
vars = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

# Statistics per timestep

In [24]:
def calc_stats_timesteps(temp, radius=300, season='year'):
    params = gpd.read_parquet(f'../data/processed_data/processed_station_params_{radius}.parquet')
    params_merged = gpd.read_parquet(f'../data/processed_data/processed_station_params_merged_blocks_{radius}.parquet')
    params = params.set_index('station_id')
    params_merged = params_merged.set_index('station_id')
    to_remove = ['station_no','station_name','station_long_name','station_type','station_lat','station_lon','mounting_structure','dominant_land_use','local_climate_zone','urban_atlas_class','urban_atlas_code','geometry','SVF_3D']
    params = params.drop(to_remove, axis=1)
    params['HW_del_merged'] = params['BuHt_wmean'] / params_merged['BuW_delaunay_mean']
    params['BuW_delaunay_mean_merged'] = params_merged['BuW_delaunay_mean']
    params['BuERI_mode_merged'] = params_merged['BuERI_mode']
    vars = params.columns
    stats_dict = {}

    vars = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum', 'BuIBD', 'BuAdj', 'BuSWR_3D_median', 'BuHt_wmean','StrHW_median','SVF_3D_mean','SVF_3D_std', 'BuERI_mode', 'StrClo400_median','BuAre_median','BuVol_3D_median','BuEWA_3D_median','BuSWR_median','BuHt_max','StrHW_mean','BuERI_wmean']
    vars = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
    vars = ['BuW_delaunay_mean_merged','HW_del_merged', 'BuERI_mode_merged']
    for var in vars:
        params_v = params[[var]]
        print(params_v)
        for time in temp.columns:

            print(f'Processing {var} for {time}')
            params_t = params_v.merge(temp[time], left_on='station_id', right_on='station_id',how='inner')
            print(params_t)
            stats = main.calculate_statistics(params_t, time)
            stats.index = stats['Parameter']
            stats['Time'] = time
            stats_dict[time] = stats
        pd.concat(stats_dict).to_csv(f'../data/processed_data/2024/stats_timesteps_{var}_{season}_{radius}.csv')

In [ ]:
calc_stats_timesteps(temp, radius=300, season='year')

In [62]:
vars = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum', 'BuIBD', 'BuAdj', 'BuSWR_3D_median', 'BuHt_wmean','StrHW_median','SVF_3D_mean','SVF_3D_std', 'BuERI_mode', 'StrClo400_median','BuAre_median','BuVol_3D_median','BuEWA_3D_median','BuSWR_median','BuHt_max','StrHW_mean','BuERI_wmean']

### Spider Plot data

In [7]:
radius=300

In [14]:
vars = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

In [ ]:
lon = 7.85222
lat = 47.9959

for j in vars:

    stats_dict = pd.read_csv(f"../data/processed_data/2024/stats_timesteps_{j}_year_{radius}.csv")
    stats_dict['Time']=pd.to_datetime(stats_dict['Time'])

    times = {}

    stats_dict['Time'] = pd.to_datetime(stats_dict['Time'])

    for i in stats_dict['Time'].unique():

        times[i] = get_times(i, lon, lat)

    stats_dict['sunrise'] = stats_dict['Time'].apply(
        lambda x: times[x]['sunrise'].replace(tzinfo=timezone.utc))
    stats_dict['sunset'] = stats_dict['Time'].apply(
        lambda x: times[x]['sunset'].replace(tzinfo=timezone.utc))

    stats_dict['time_of_day'] = np.where(
        stats_dict['Time'] < stats_dict['sunrise'], 'night',
        np.where(stats_dict['Time'] > stats_dict['sunset'], 'night', 'day'))
    
    stats_dict = stats_dict[stats_dict['time_of_day'].isin(['night'])]

    p_values = stats_dict['Spearman p-value'].values

    # Run FDR correction (Benjamini-Hochberg, common default)
    reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

    stats_dict['Spearman p-value corrected'] = pvals_corrected
    #stats_dict  = stats_dict[(stats_dict['fraction_of_night'] < 1)]
    print('Mean',j,stats_dict['Spearman Correlation'].mean())
    print('Spearman corr > 0.5',j,len(stats_dict[(((stats_dict['Spearman Correlation'] > 0.5) | (stats_dict['Spearman Correlation'] < -0.5))) & (stats_dict['Spearman p-value corrected'] < 0.05)]) / len(stats_dict))
    #print('Spearman corr > 0.7',i,len(stats_dict[(stats_dict['Spearman Correlation'] > 0.7) | (stats_dict['Spearman Correlation'] < -0.7)]) / len(stats_dict))
    #print('Spearman p-val < 0.05',i,len(stats_dict[(stats_dict['Spearman p-value'] < 0.005)]) / len(stats_dict))

In [ ]:
lon = 7.85222
lat = 47.9959

for j in vars:

    stats_dict = pd.read_csv(f"/Users/lisawink/Documents/paper1/data/processed_data/2024/stats_timesteps_{j}_year_{radius}.csv")
    stats_dict['Time']=pd.to_datetime(stats_dict['Time'])

    times = {}

    stats_dict['Time'] = pd.to_datetime(stats_dict['Time'])

    for i in stats_dict['Time'].unique():

        times[i] = get_times(i, lon, lat)

    stats_dict['sunrise'] = stats_dict['Time'].apply(
        lambda x: times[x]['sunrise'].replace(tzinfo=timezone.utc))
    stats_dict['sunset'] = stats_dict['Time'].apply(
        lambda x: times[x]['sunset'].replace(tzinfo=timezone.utc))

    stats_dict['time_of_day'] = np.where(
        stats_dict['Time'] < stats_dict['sunrise'], 'night',
        np.where(stats_dict['Time'] > stats_dict['sunset'], 'night', 'day'))
    
    stats_dict = stats_dict[stats_dict['time_of_day'].isin(['night'])]

    p_values = stats_dict['Spearman p-value'].values

    # Run FDR correction (Benjamini-Hochberg, common default)
    reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

    stats_dict['Spearman p-value corrected'] = pvals_corrected
    #stats_dict  = stats_dict[(stats_dict['fraction_of_night'] < 1)]
    print('Mean',j,stats_dict['Spearman Correlation'].mean())
    print('Spearman corr > 0.5',j,len(stats_dict[(((stats_dict['Spearman Correlation'] > 0.5) | (stats_dict['Spearman Correlation'] < -0.5))) & (stats_dict['Spearman p-value corrected'] < 0.05)]) / len(stats_dict))
    #print('Spearman corr > 0.7',i,len(stats_dict[(stats_dict['Spearman Correlation'] > 0.7) | (stats_dict['Spearman Correlation'] < -0.7)]) / len(stats_dict))
    #print('Spearman p-val < 0.05',i,len(stats_dict[(stats_dict['Spearman p-value'] < 0.005)]) / len(stats_dict))